# Create Iron Sediment and Vent Forcing files

This notebook is based off an IDL script from J. Keith Moore (UCI), received Sept 11, 2024.
It can use output from a POP2 run (`POC_FLUX_IN`, `TEMP`, `UVEL`, `VVEL`, and `KVMIX`)
or output from a MOM6 run (`POC_FLUX_IN`, `thetao`, `uo`, `vo`, `Kd_itides`, `Kd_bkgnd` , and `Kd_BBL`).

This notebook runs on a Casper login node (10 GB memory) when processing 10 years of MOM6 output.

## Step 0: Python Imports

In [ ]:
import datetime
import glob
import os

from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

import iron_utils

## Step 1: Read in Data

Users should modify this first cell to tune parameters, choose a grid, and determine what netCDF files (if any) should be written.

In [ ]:
# User configurable parameters
xfactor = 0.005639
minval = 0.1
minoxic = 0.0005343  # units: mmol m-2 yr-1
minpoc = 5.          # Applied to top 2000 m; units: g m-2 yr-1
total_ventflux = 5   # units: gmol yr-1

# grid = 'tx2_3v2'
grid = 'tx2_3v3'

write_output = True
# write_output = False
write_percsed = False
verbose = False   # true will add output in nearest-neighbor search

When adding a new grid, users should modify this if statement to define the provenance of the output.
Note that runs interpolated from POP expect output on the new grid in `{iron_utils.dirwork}/*.POP_{POP_grid}_to_MOM_{grid}_new.zarr`,
while output from MOM6 is found based on the case name and the user name of the individual who created / ran the case.

In [ ]:
if grid == 'tx2_3v2':
    interp_from_pop = True
    POP_grid = 'gx1v7'
    # Location of SCRIP grid file (used to get area of new grid)
    scrip_file = os.path.join(os.path.sep,
                             'glade',
                             'campaign',
                             'cesm',
                             'cesmdata',
                             'inputdata',
                             'share',
                             'scripgrids',
                             'tx2_3_SCRIP_230415.nc')
    # SCRIP file is in sq rad, notebook wants m^2
    area_conv = 6.371e6**2
    # Location of geometry file (needed to get depth)
    geom_file = os.path.join(os.sep,
                             'glade',
                             'work',
                             'mlevy',
                             'cesm_inputdata',
                             'geom_files',
                             'tx2_3v2.ocean_geometry.nc')

    mesh_file = 'tx2_3v2_230415_ESMFmesh.nc'
elif grid == 'tx2_3v3':
    # already mks
    interp_from_pop = False

    # variables to help us find files
    run_user = 'kristenk'
    case = 'g.e30_a09.GW1850MARBL_JRA.TL319_t233_wgx3_hycom1_N75.023'
    start_year = 9
    end_year = 10

    mesh_file = 'tx2_3v3_260305_ESMF_mesh.nc'
else:
    raise ValueError(f'"{grid}" is not a recognized grid')

In [ ]:
# Expect cgs output if interpolating from POP, and also read different variables
if interp_from_pop:
    # cgs -> mks
    m2depth = 100.
    pop_var_list = ['POC_FLUX_IN', 'UVEL', 'VVEL', 'KVMIX', 'TEMP']
else:
    m2depth = 1.
    bgc_var_list = ['POC_FLUX_IN']
    physics_var_list = ['thetao', 'uo', 'vo', 'Kd_bkgnd' , 'Kd_itides', 'Kd_BBL']

# Set up global attributes to help future users determine how forcing files were created
attrs = {}
attrs['history'] = 'Created by running bgc/iron_forcings/gen_fesedflux_files.ipynb found in github.com/NCAR/tx2_3'
attrs['creation_date'] = str(datetime.date.today())
attrs['author'] = os.environ['USER']

# Set up output file location / names
out_dir = './output'
today = ''.join(str(datetime.date.today()).split('-'))[2:] # YYMMDD
percsedfile_out = os.path.join(out_dir, f'percentsed_{grid}_cesm2_ecos1.1_2024_{today}.nc')
flux_out = os.path.join(out_dir, f'fefluxes_sed2024algo_vent2026algo_{grid}.c{today}.nc')

# If writing output, delete existing version of the file
if write_output:
    filelist = [flux_out]
    if write_percsed:
        filelist.append(percsedfile_out)
        
    for file in filelist:
        print(file)
        if os.path.exists(file):
            print(f'{file} exists, removing and it will be recreated')
            os.remove(file)

In [ ]:
try:
    # Some grids do not have model output
    # But I have saved the ocean_geometry stream
    ds_geom = xr.open_dataset(geom_file)
except:
    archive_dir = os.path.join(os.path.sep,
                              'glade',
                              'derecho',
                              'scratch',
                              run_user,
                              'archive',
                              case,
                              'ocn',
                              'hist')
    ds_geom = xr.open_dataset(os.path.join(archive_dir, f'{case}.mom6.h.ocean_geometry.nc'))
depth = ds_geom['D']

esmf_mesh_dir = os.path.join(os.path.sep,
                             'glade',
                             'campaign',
                             'cesm',
                             'cesmdata',
                             'inputdata',
                             'share',
                             'meshes',
                            )
ds_mesh = xr.open_dataset(os.path.join(esmf_mesh_dir, mesh_file))
ds_mesh

In [ ]:
# Create empty datasets to collect data arrays
local_vars = xr.Dataset()
outputs = xr.Dataset()

In [ ]:
%%time

if interp_from_pop:
    # Expect POP variable names to be written out, but this notebook does everything in MOM6 variable space
    rename_dict = {'TEMP': 'thetao',
                   'UVEL': 'uo',
                   'VVEL': 'vo',
                   'nlon': 'xh',
                   'nlat': 'yh',
                   'z_t': 'z_l',
                   'z_w_bot': 'z_i',
                   'TLONG': 'geolon',
                   'TLAT': 'geolat',
                  }

    # Read temporal means of input fields
    ds_means = xr.open_mfdataset(os.path.join(iron_utils.dirwork, f'*.POP_{POP_grid}_to_MOM_{grid}_new.zarr'),
                                 compat='override',
                                 engine='zarr')[pop_var_list].squeeze().compute().drop_vars('time').rename(rename_dict)
    area_arr = xr.open_dataset(scrip_file)['grid_area'].data.reshape((480,540)) * area_conv
else:
    archive_dir = os.path.join(os.path.sep,
                              'glade',
                              'derecho',
                              'scratch',
                              run_user,
                              'archive',
                              case,
                              'ocn',
                              'hist')

    # Area, lat, and lon comes from mom6.h.static
    ds_static = xr.open_dataset(os.path.join(archive_dir, f'{case}.mom6.h.static.nc'))
    area_arr = ds_static['areacello'].data

    # Global mean file
    means_file = os.path.join(iron_utils.dirwork,
                              f'{case}.means_{start_year:04d}_through_{end_year:04d}.nc')

    if not os.path.isfile(means_file):
        # Create 10-year means from MOM6 output
        bgc_file_list = []
        physics_file_list = []
        for year in range(start_year,end_year+1):
            print(f'Processing year {year:04d}')
            physics_file_list = physics_file_list + sorted(glob.glob(os.path.join(archive_dir, f'{case}.mom6.h.z.{year:04d}-??.nc')))
            bgc_file_list = bgc_file_list + sorted(glob.glob(os.path.join(archive_dir, f'{case}.mom6.h.bgc.z.{year:04d}-??.nc')))

        print(f'Found {len(physics_file_list)} physics stream files and {len(bgc_file_list)} bgc stream files')
        ds_tmp = xr.merge(
            [xr.open_mfdataset(physics_file_list, decode_times=False, compat='override', data_vars='minimal', coords='minimal')[physics_var_list].mean('time'),
            xr.open_mfdataset(bgc_file_list, decode_times=False, compat='override', data_vars='minimal', coords='minimal')[bgc_var_list].mean('time')],
            compat='override')
        print('Finished merging datasets')
        ds_tmp.compute().to_netcdf(means_file, format="NETCDF3_64BIT")
        print('Created netCDF file!')

    attrs['provenance'] = f'Generated from years {start_year:04d} through {end_year:04d} of {case}, run by {run_user}'
    ds_means = xr.open_dataset(means_file).isel(z_i=slice(1,None)).assign_coords({'geolat': ds_static['geolat'], 'geolon': ds_static['geolon']})

# mask out land in area_arr
area_arr = np.where(np.isfinite(ds_means['thetao'].isel(z_l=0).data), area_arr, np.nan)
print(f'Total area of the ocean is {np.nansum(area_arr):.3e} m^2')

In [ ]:
ds_means['thetao'].isel(z_l=0).plot()

In [ ]:
ds_means['mask'] = xr.zeros_like(ds_means['thetao'], dtype=bool)
ds_means['mask'].name = 'Ocean Mask'
ds_means['mask'].data = np.where(np.isfinite(ds_means['thetao'].data), True, False)
# Temp had some unexpected values of -1 in deep ocean for blocks where TLAT and TLON are missing (LBE to blame)
ds_means['mask'].data = np.where(np.logical_not(ds_means['mask'].isel(z_l=0).data), False, ds_means['mask'].data)
ds_means['land_mask'] = xr.zeros_like(ds_means['mask'])
ds_means['land_mask'].name = 'Land Mask'
ds_means['land_mask'].data = np.logical_not(ds_means['mask'].data)
ds_means

## Step 2: Compute Mean Horizontal Speed

Looks like Keith used $\ell_1$ norm here

In [ ]:
ds_means['velocity'] = xr.zeros_like(ds_means['uo'])
ds_means['velocity'].name = 'velocity'
ds_means['velocity'].attrs['units'] = 'm/s'
# Note: IDL script loops through popz-2, not popz-1 => velocity at bottom is 0!
ds_means['velocity'].data[:-1,:,:] = (np.abs(ds_means['uo'].data[:-1,:,:]) + np.abs(ds_means['vo'].data[:-1,:,:])) / m2depth
ds_means['velocity'].isel(z_l=0).plot()

## Step 3: Minimum percent sed when land-adjacent

Also, rescale in vertical

In [ ]:
# Set up arrays of indices for cell to left / right
left_ind = np.arange(ds_means['mask'].sizes['xh']) - 1
left_ind[0] = ds_means['mask'].sizes['xh'] - 1
right_ind = np.arange(ds_means['mask'].sizes['xh']) + 1
right_ind[-1] = 0

# set up arrays of indices for cells two below, directly below, directly above, and two above
down2_ind = np.arange(ds_means['mask'].sizes['yh']) - 2
down2_ind[:2] = 0
down_ind = np.arange(ds_means['mask'].sizes['yh']) - 1
down_ind[0] = 0
up_ind = np.arange(ds_means['mask'].sizes['yh']) + 1
up_ind[-1] = ds_means['mask'].sizes['yh'] - 1
up2_ind = np.arange(ds_means['mask'].sizes['yh']) + 2
up2_ind[-2:] = ds_means['mask'].sizes['yh'] - 1

In [ ]:
ds_means['land_adj'] = xr.zeros_like(ds_means['mask'])
ds_means['land_adj'].name = "Land Adjacent"

# look for land due east/west of cell
for lon_ind in [left_ind, right_ind]:
    ds_means['land_adj'].data = np.where(ds_means['land_mask'].data[:,:,lon_ind],True, ds_means['land_adj'].data)

# look for land due north/south of cell
for lat_ind in [down2_ind, down_ind, up_ind, up2_ind]:
    ds_means['land_adj'].data = np.where(ds_means['land_mask'].data[:,lat_ind,:], True, ds_means['land_adj'].data)

# # look for land in the corners of the halo
for lat_ind in [down2_ind, down_ind, up_ind, up2_ind]:
    for lon_ind in [left_ind, right_ind]:
        ds_means['land_adj'].data = np.where((ds_means['land_mask'].data[:,lat_ind,:])[:,:,lon_ind],True, ds_means['land_adj'].data)

# Actual land points are not considered land-adjacent
ds_means['land_adj'].data = np.where(ds_means['land_mask'].data, False, ds_means['land_adj'].data)

# Plot land-adjacent cells in surface layer
print(f"There are {np.sum(ds_means['land_adj'].isel(z_l=0).data)} land-adjacent cells in the top level")
ds_means['land_adj'].isel(z_l=0).plot()
# ds_means['land_adj'].isel(z_l=-1).plot()

In [ ]:
# Setting dimensions to match temperature, but data comes from ds_percent_sed
outputs['percsed'] = xr.zeros_like(ds_means['thetao']).rename({'z_l': 'DEPTH',
                                                             'yh': 'ny',
                                                             'xh': 'nx'})
outputs['percsed'].name = "Percent Sed"
outputs['percsed'].attrs = {}
outputs['percsed'].encoding['_FillValue'] = None

# percsed should be 1 in the bottom-most active ocean layer, and 0 elsewhere
for k in range(outputs['percsed'].sizes['DEPTH']):
    outputs['percsed'].data[k,:,:] = np.where(ds_means['mask'].sum('z_l').data == k+1, 1, 0)

# Two steps in land-adjacent points:
# 1. Set percsed to max(minval, percsed)
outputs['percsed'].data = np.where(ds_means['land_adj'].data,
                                   np.maximum(minval, outputs['percsed'].data),
                                   outputs['percsed'].data)
# 2. Normalize percsed so sum of every column is 1
percsed_sum = np.where(ds_means['land_adj'].data, outputs['percsed'].sum('DEPTH'), 1)

# Note that perced_sum = 1 when not land-adjacent, so no np.where() statement needed
outputs['percsed'].data = outputs['percsed'].data / percsed_sum.data

# Add units to depth and remove fill value from coordinates
outputs = outputs.assign_coords(DEPTH=ds_means['z_l'].data / m2depth)
outputs['DEPTH'].attrs['units'] = 'm'
outputs['DEPTH'].attrs['edges'] = 'DEPTH_EDGES'

for coord in outputs.coords:
    outputs[coord].encoding['_FillValue'] = None

outputs['percsed'].sum('DEPTH').plot()

In [ ]:
# Write to File
if write_output and write_percsed:
    outputs['percsed'].to_dataset(name="PERCENTSED").to_netcdf(percsedfile_out)

## Step 4: Compute Sediment Input from Oxic Sediments

First enforce $0.1 \textrm{ cm/s} \le \textrm{speed} \le 3.3 \textrm{ cm/s}$, and then set speed to 0 anywhere percsed is 0.
Note that speed is stored in m/s in the python, and the min and max are applied in the correct units.

In [ ]:
local_vars['speed'] = xr.zeros_like(ds_means['velocity'])
local_vars['speed'].name = 'Local copy of speed'
local_vars['speed'].data = np.minimum(0.033, np.maximum(0.001, ds_means['velocity'].data)) # MNL: 0.2 -> 0.02; also units are now m/s

# Above we restrict speed to between 0.001 m/s and 0.033 m/s; shallower than 1000 m want minimum speed to be 0.01 m/s 
depth_thres = 1000. * m2depth
min_speed = 0.01
local_vars['speed'].data = np.where(np.logical_and(local_vars['speed'] < min_speed, ds_means['velocity'].z_l < depth_thres),
                                    min_speed,
                                    local_vars['speed'].data
                                   )

local_vars['speed'].data = np.where(outputs['percsed'].data > 0., local_vars['speed'].data, 0.)
local_vars['speed'].isel(z_l=0).plot()

We then compute the sum of the diffusivity coefficients from tidal and background mixing
[and BBL mixing, if available] ($K_d$, in units of $\mathrm{m}^2/\mathrm{s}$), applying a global maximum of $3.3*10^{-5}$ $\mathrm{m}^2/\mathrm{s}$.

The scale factor is the bigger value of speed in cm/s and $K_d$, in $\mathrm{cm}^2/\mathrm{s}$

In [ ]:
local_vars['scale'] = xr.zeros_like(local_vars['speed'])
local_vars['scale'].name = 'Scale Factor'

if interp_from_pop:
    local_vars['Kd'] = xr.zeros_like(ds_means['KVMIX'])
    local_vars['Kd'].name = 'Kd'
    Kd_loc = ds_means['KVMIX'].data[1:-1,:,:]
else:
    local_vars['Kd'] = xr.zeros_like(ds_means['Kd_itides'])
    local_vars['Kd'].name = 'Kd'
    Kd_loc = ds_means['Kd_itides'].isel(z_i=slice(1,-1)).data + ds_means['Kd_bkgnd'].isel(z_i=slice(1,-1)).data + ds_means['Kd_BBL'].isel(z_i=slice(1,-1)).data

# If Kd was in cgs, it is converted to mks in this step
local_vars['Kd'].data[2:,:,:] = np.where(outputs['percsed'].data[2:,:,:] > 0., Kd_loc, 0.) / (m2depth*m2depth)
local_vars['Kd'].data = np.minimum(local_vars['Kd'].data, 3.3e-5)
local_vars['scale'].data = np.maximum(local_vars['Kd'].data * 1e4, local_vars['speed'].data * 100.)  # original code used cgs, not mks

The initial iron sediment flux is the product of minoxic, percsed, and the scale factor computed above.
It is converted from mmol/m$^2$/yr to $\mu$mol/m$^2$/d and then the land mask is applied.

In [ ]:
outputs['fesed'] = xr.zeros_like(outputs['percsed'])
outputs['fesed'].name = "Fe Sediment"
outputs['fesed'].data = minoxic * outputs['percsed'].data * local_vars['scale'].data
# Convert to model units
outputs['fesed'].data = outputs['fesed'].data / (365. * 1e-3)
outputs['fesed'].attrs = {'regrid_method': 'conservative',
                          'units': 'micromol/m^2/d',
                          'long_name': 'Fe Sediment Flux'}

# mask out land
outputs['fesed'].data = np.where(ds_means['mask'].isel(z_l=0).data, outputs['fesed'].data, np.nan)
outputs['fesed'].encoding['_FillValue'] = 1e-20

We then construct the final output field.
For MOM6 we need coordinate fields named both DEPTH and DEPTH_EDGES.

In [ ]:
ds_out = outputs['fesed'].to_dataset(name='FESEDFLUX').assign_coords(DEPTH_EDGES=np.insert(ds_means['z_i'].data/m2depth,0,0))
del(ds_out['geolon'].encoding['missing_value'])
del(ds_out['geolat'].encoding['missing_value'])
ds_out['DEPTH_EDGES'].attrs['units'] = 'm'
ds_out['DEPTH_EDGES'].encoding['_FillValue'] = None
ds_out.attrs = attrs
ds_out

One final scale factor is applied.
For output interpolated from POP2, we want to ensure the global 3D integral is unchanged from a previous version of this file.
For output generated by MOM6, we want to ensure the horizontal integral of the vertical sum is a given value.

In [ ]:
fesed_old = xr.open_dataset('/glade/campaign/cesm/cesmdata/inputdata/ocn/mom/tx2_3v2/fesedflux_2024algo_tx2_3v2.c251229.nc')
if grid == 'tx2_3v2':
    print(np.nanmax(np.abs(ds_out['FESEDFLUX'].data - fesed_old['FESEDFLUXIN'].data)))

In [ ]:
def compute_global_mean(ds, varname, area_arr):
    weights = (ds['DEPTH_EDGES'].data[1:,None,None] - ds['DEPTH_EDGES'].data[:-1,None,None]) * area_arr
    return np.nansum(ds[varname].data * weights) / np.nansum(weights)

def compute_global_mean_vert_sum(ds, varname, area_arr):
    return compute_global_int_vert_sum(ds, varname, area_arr) / np.nansum(area_arr)

def compute_global_int_vert_sum(ds, varname, area_arr):
    return np.nansum(ds[varname].sum('DEPTH').data * area_arr)

if grid == 'tx2_3v2':
    old_mean = compute_global_mean(fesed_old, 'FESEDFLUXIN', area_arr)
    new_mean = compute_global_mean(ds_out, 'FESEDFLUX', area_arr)
    print(f'Global mean from previous run: {old_mean}')
    print(f'Current global mean: {new_mean}')
    scalef   = old_mean / new_mean
else:
    new_int = compute_global_int_vert_sum(ds_out, 'FESEDFLUX', area_arr)
    target_int = 1e12
    print(f'Current horizontal integral of vertical sum: {new_int:.3e}')
    print(f'Target horizontal integral of vertical sum: {target_int:.3e}')
    scalef = target_int / new_int

print(f'scale factor: {scalef}')

In [ ]:
ds_out['FESEDFLUX'].data = ds_out['FESEDFLUX'].data * scalef

if grid == 'tx2_3v2':
    print(np.nanmax(np.abs(ds_out['FESEDFLUX'].data - fesed_old['FESEDFLUXIN'].data)))

In [ ]:
# Vertically summed sediment flux
fesed_vertsum = ds_out['FESEDFLUX'].sum('DEPTH')
fesed_vertsum.plot()
print(f"global integral (of vertical sum): {compute_global_int_vert_sum(ds_out, 'FESEDFLUX', area_arr):7.3e}")
print(f"global mean (of vertical sum): {compute_global_mean_vert_sum(ds_out, 'FESEDFLUX', area_arr):7.3e}")

In [ ]:
# Vertically integrated sediment flux
fesed_vertsum = (ds_out['FESEDFLUX']*ds_out['DEPTH']).sum('DEPTH')
fesed_vertsum.plot()
print(f"global mean: {compute_global_mean(ds_out, 'FESEDFLUX', area_arr):7.3e}")

## Step 5: Compute Sediment Input from Reducing Sediments

Computation is based on POC_FLUX_IN from previous run, but amplified in a few regions.
Comments in the next notebook cell specify the precise coordinates of these regions.

1. POC is scaled by either 5x in some portions of the shallow western Pacific (depth <= 400 m) and 2.5x in others
2. POC is scaled by 2.5x in the shallow Southern Ocean (depth <= 400m, latitudes south of 56.2° S)
3. A minimum value of 5 g/m$^2$/yr is applied in the top 2000 m

In [ ]:
# convert POC_FLUX_IN from mmol / m^3  [length units] / s -> g / m^2 / yr
# 365 * 86400 s / yr, 12.011 gC / mol C, 1 m / {1 m or 100 cm}, 0.001 mmol / mol
local_vars['POC'] = xr.zeros_like(ds_means['POC_FLUX_IN'])
local_vars['POC'].data = (365. * 86400.) * 12.011 *(1/m2depth) * 0.001 * ds_means['POC_FLUX_IN'].data

# Function to define mask given latitude and longitude ranges
def region_mask_from_grid(lats, lons, lat_range, lon_range):
    return np.logical_and(np.logical_and(lats > lat_range[0], lats < lat_range[1]),
                          np.logical_and(lons > lon_range[0], lons < lon_range[1]))

# Update Western Pacific
# IDL comment: WPac (25S-0,140-235E, 0-6N,0-20S x 10.0 0-504m)
# Actual range: 5x change in roughly 116E - 240E, 2S - 6N
#               2.5x change in roughly 140 E - 240 E, 20S - 2S
#               Depth limited to ~400m
z_l_max = np.argmax(local_vars['z_l'].data > 400*m2depth)
print(f'z_l_max = {z_l_max} will include depth of {local_vars['z_l'].data[z_l_max-1]/m2depth} but not {local_vars['z_l'].data[z_l_max]/m2depth}')
local_vars['POC'].data[:z_l_max,:,:] = np.where(region_mask_from_grid(ds_means['geolat'].data, ds_means['geolon'].data, (-2.01, 6.1), (115.8, 239.6)),
                                                local_vars['POC'].data[:z_l_max,:,:] * 5.,
                                                local_vars['POC'].data[:z_l_max,:,:])

local_vars['POC'].data[:z_l_max,:,:] = np.where(region_mask_from_grid(ds_means['geolat'].data, ds_means['geolon'].data, (-20.2, -2.25), (140.5, 239.6)),
                                                local_vars['POC'].data[:z_l_max,:,:] * 2.5,
                                                local_vars['POC'].data[:z_l_max,:,:])
# Update Southern Ocean
# Depth limited to ~600 m
# (After looking at bathymetry maps, increasing from 400m to 600m adds a lot more of the shelf in Weddell Sea)
z_l_max = np.argmax(local_vars['z_l'].data > 600*m2depth)
print(f'z_l_max = {z_l_max} will include depth of {local_vars['z_l'].data[z_l_max-1]/m2depth} but not {local_vars['z_l'].data[z_l_max]/m2depth}')

# 1.5x change south of 35S
local_vars['POC'].data[:z_l_max,:,:] = np.where(ds_means['geolat'].data < -35,
                                                local_vars['POC'].data[:z_l_max,:,:] * 1.5,
                                                local_vars['POC'].data[:z_l_max,:,:])

# additional 2x change south of 56 S
# (changed from 2.5x when additional scaling from 35 S was introduced
local_vars['POC'].data[:z_l_max,:,:] = np.where(ds_means['geolat'].data < -56.2,
                                                local_vars['POC'].data[:z_l_max,:,:] * 2,
                                                local_vars['POC'].data[:z_l_max,:,:])

# Apply minimum value to top 2000m
z_l_max = np.argmax(local_vars['z_l'].data >= 2000*m2depth)
print(f'z_l_max = {z_l_max} will include depth of {local_vars['z_l'].data[z_l_max-1]/m2depth} but not {local_vars['z_l'].data[z_l_max]/m2depth}')
local_vars['POC'].data[:z_l_max,:,:] = np.where(local_vars.isel(z_l=slice(0,z_l_max))['POC'].data < minpoc, minpoc, local_vars['POC'].data[:z_l_max,:,:])

There is a temperature dependence term as well.
Given $T$ in °C, and with a reference temperature $T_{ref}$ of 32°

Tfunc = $1.5^{\frac{T - T_ref}{10}}$

In [ ]:
local_vars['Tfunc'] = xr.zeros_like(ds_means['thetao'])
local_vars['Tfunc'].name = 'Tfunc'
local_vars['Tfunc'].data = 1.5**((ds_means['thetao'].data - 32.0) / 10.)

In [ ]:
outputs['fesedRed'] = xr.zeros_like(outputs['fesed'])
outputs['fesedRed'].name = 'Iron Sediment Reduced'

outputs['fesedRed'].data = local_vars['POC'].data * xfactor * outputs['percsed'].data * local_vars['scale'].data * local_vars['Tfunc'].data

# Remove fill value below ocean floor (keep continents masked)
outputs['fesedRed'].data = np.where(np.logical_and(ds_means['mask'].isel(z_l=0).data, np.isnan(outputs['fesedRed'].data)), 0., outputs['fesedRed'].data)
outputs['fesedRed'].encoding['_FillValue'] = 1e20

# Convert to model units
outputs['fesedRed'].data = outputs['fesedRed'].data / (365. * 1e-3)

In [ ]:
ds_out['FEREDSEDFLUX'] = xr.zeros_like(ds_out['FESEDFLUX'])
ds_out['FEREDSEDFLUX'].data = outputs['fesedRed'].data
ds_out['FEREDSEDFLUX'].attrs['long_name'] = 'Fe Red Sediment Flux'

# fesedRed_output = outputs['fesedRed'].to_dataset(name='FESEDFLUXIN').assign_coords(DEPTH_EDGES=np.insert(ds_means['z_i'].data/m2depth,0,0))
# fesedRed_output['DEPTH_EDGES'].attrs['units'] = 'm'
# fesedRed_output['DEPTH_EDGES'].encoding['_FillValue'] = None
# fesedRed_output.attrs = attrs
# fesedRed_output

In [ ]:
fesedRed_vertsum = ds_out['FEREDSEDFLUX'].sum('DEPTH')
fesedRed_vertsum.plot(vmax=2)
print(f"global integral (of vertical sum): {compute_global_int_vert_sum(ds_out, 'FEREDSEDFLUX', area_arr):7.3e}")
print(f"global mean (of vertical sum): {compute_global_mean_vert_sum(ds_out, 'FEREDSEDFLUX', area_arr):7.3e}")

### Step 6: Compute Iron Vent Flux

We want to find the ocean grid cell nearest to each vent from the obs product.
The vent is placed 300 meters above the sea floor.
Once all the vents are placed, each has the same constant flux such that
the global integral of the flux is a specified value (e.g. 5 gmol / year).

To find the nearest neighbor, we convert from lat-lon coordinates to Cartesian space
and then find the grid cell that maximizes the dot product with the vent location.

In [ ]:
def convert_lat_lon_to_xyz(lon_array_deg, lat_array_deg):
    """ Returns numpy arrays for (x, y, z) """
    deg_to_rad = (np.pi / 180.0)
    lon_array_rad = lon_array_deg * deg_to_rad
    lat_array_rad = lat_array_deg * deg_to_rad
    x = np.cos(lon_array_rad) * np.cos(lat_array_rad)
    y = np.sin(lon_array_rad) * np.cos(lat_array_rad)
    z = np.sin(lat_array_rad)
    return x,y,z

In [ ]:
ds_mesh['x'] = xr.zeros_like(ds_mesh['elementArea'])
ds_mesh['x'].attrs = {}
ds_mesh['y'] = xr.zeros_like(ds_mesh['x'])
ds_mesh['z'] = xr.zeros_like(ds_mesh['x'])

ds_mesh['x'].data, ds_mesh['y'].data, ds_mesh['z'].data = convert_lat_lon_to_xyz(ds_mesh['centerCoords'].isel(coordDim=0).data, ds_mesh['centerCoords'].isel(coordDim=1).data)

for var in ['x', 'y', 'z']:
    print(ds_mesh[var].min().data, ds_mesh[var].max().data)

InterRidge gives a list of latitudes and longitudes for 721 vents.
Some sites have a minimum depth, but that field is only provided for 246 of the vents.

For now we ignore the InterRidge depth,
and place all vents a constant distance above the model grid sea floor.
Future work could use a high-resolution ocean depth dataset to better estimate vent depth.

In [ ]:
obs_data = pd.read_csv(os.path.join('inputs', 'vent_fields_all_20200325cleansorted.csv'))
for col in obs_data.columns:
    print(col)

In [ ]:
# longitudes are given in [-180,180] but MOM6 grid is -287, 73]
# The second where statement should not change any longitudes, but is included for completeness
# in case future datasets use a different longitude range
obs_data['Longitude'] = np.where(obs_data['Longitude'] >= 73, obs_data['Longitude'] - 360., obs_data['Longitude'])
obs_data['Longitude'] = np.where(obs_data['Longitude'] < -287, obs_data['Longitude'] + 360., obs_data['Longitude'])
obs_data['Longitude'].min(), obs_data['Longitude'].max()

In [ ]:
obs_data['x'], obs_data['y'], obs_data['z'] = convert_lat_lon_to_xyz(obs_data['Longitude'], obs_data['Latitude'])

for var in ['x', 'y', 'z']:
    print(np.min(obs_data[var]), np.min(obs_data[var]))

In [ ]:
%%time

inds = []
for n,(x,y,z,lon,lat) in enumerate(zip(obs_data['x'], obs_data['y'], obs_data['z'], obs_data['Longitude'], obs_data['Latitude'])):
    dotprod = x*ds_mesh['x'].data + y*ds_mesh['y'].data + z*ds_mesh['z'].data
    ind = dotprod.argmax()
    inds.append(ind)
    if verbose:
        print(f'{n}:')
        print(f'   vent lat, lon: ({lat:.3f}, {lon:.3f})')
        print(f'   nearest grid:  ({ds_mesh["centerCoords"].isel(coordDim=1).data[ind]:.3f}, {ds_mesh["centerCoords"].isel(coordDim=0).data[ind]:.3f})')
        print(f'   Cartesian details:')
        print(f'      vent loc: ({x:.3f}, {y:.3f}, {z:.3f})')
        print(f'      nearest cell: ({ds_mesh["x"].data[ind]:.3f}, {ds_mesh["y"].data[ind]:.3f}, {ds_mesh["z"].data[ind]:.3f})')
        print(f'      max dotprod {dotprod[ind]:.6f}')
        if n<len(obs_data['x']):
            print('----')

In [ ]:
ds_tmp = xr.zeros_like(depth).to_dataset(name='vent_loc')
ds_tmp['vent_loc'].attrs = {}

vent_loc = np.zeros_like(ds_mesh['x'].data)
vent_loc[inds] = 1
print(f'There are {np.sum(vent_loc)} vents before masking out land')

vent_loc = np.where(ds_mesh['elementMask'] == 1, vent_loc, np.nan)
print(f'There are {np.nansum(vent_loc)} vents after masking out land')

vent_loc = np.where(np.isfinite(vent_loc), vent_loc, 0.0625)
ds_tmp['vent_loc'].data = vent_loc.reshape((480, 540))
ds_tmp['vent_loc'].plot(cmap='binary')

vent_loc = np.where(vent_loc > 0.1, vent_loc, 0.0)
ds_tmp['vent_loc'].data = np.where(ds_tmp['vent_loc'].data > 0.1, ds_tmp['vent_loc'].data, 0.)

Compute the total horizontal area of cells containing a vent
(this value is used to find the per-vent flux):

In [ ]:
tot_area = np.nansum(ds_geom['Ah'].data*ds_tmp['vent_loc'].data)
cell_ventflux = (total_ventflux / tot_area) * (1e15 / 365)
cell_ventflux

In [ ]:
ds_out['FEVENTFLUX'] = xr.zeros_like(ds_out['FESEDFLUX'])
ds_out['FEVENTFLUX'].attrs['long_name'] = 'Fe Vent Flux'

# Setup 2D of target depths, and put cell_ventflux at correct depth
# Note: this ignores depth provided by observations!
target_depth = np.maximum(depth.data - 300., 0.)
for k in range(len(ds_out['DEPTH'])):
    ds_out['FEVENTFLUX'].data[k,:,:] = np.where(np.logical_and(target_depth >= ds_out['DEPTH_EDGES'].data[k],
                                                                target_depth < ds_out['DEPTH_EDGES'].data[k+1]),
                                                 cell_ventflux*ds_tmp['vent_loc'].data,
                                                 0.
                                                )
ds_out['FEVENTFLUX'].data = np.where(np.isfinite(ds_out['FESEDFLUX'].data), ds_out['FEVENTFLUX'].data, np.nan)
modeled_total_ventflux = np.nansum(ds_geom['Ah'].data*ds_out['FEVENTFLUX'].sum('DEPTH').data) * 365 * 1e-15
print(f'Total ventflux from file is {modeled_total_ventflux}, which differs from {total_ventflux} by {modeled_total_ventflux - total_ventflux:.3e}')

### Step 7: Write output file

In [ ]:
for coord in ds_out.coords:
    ds_out[coord].encoding['_FillValue'] = None

for var in ds_out.data_vars:
    ds_out[var].encoding['_FillValue'] = 1e20

ds_out

In [ ]:
if write_output:
    print(f'Writing {flux_out}')
    ds_out.to_netcdf(flux_out, format="NETCDF3_64BIT")